In [1]:
import tensorflow as tf

2025-06-17 13:21:44.677247: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-17 13:21:44.703360: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-17 13:21:44.892947: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-17 13:21:45.033347: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750166505.161910   35964 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750166505.19

In [4]:
a = tf.keras.layers.LSTMCell(32)

In [ ]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPool2D


model = Sequential(layers=[
    Conv2D(32, (3, 3), input_shape=(64, 64, 3)),
    MaxPool2D(pool_size=(3, 3), strides=(2, 2))
])

for layer in model.layers:
    print(layer.output_shape)

In [15]:
from migration.models.mlp import get_mlp

In [16]:
a = get_mlp([4,6])

In [6]:
a.state_size

[32, 32]

In [8]:
from tensorflow.compat.v1.nn.rnn_cell import LSTMCell as LSTMCell2

In [9]:
b = LSTMCell2(32)

/workspaces/GeoTrackNet/migration/packages/v2/.venv/lib/python3.12/site-packages/tensorflow/python/keras/layers/legacy_rnn/rnn_cell_impl.py:910: UserWarning: `tf.nn.rnn_cell.LSTMCell` is deprecated and will be removed in a future version. This class is equivalent as `tf.keras.layers.LSTMCell`, and will be replaced by that in Tensorflow 2.0.
  warnings.warn("`tf.nn.rnn_cell.LSTMCell` is deprecated and will be "


In [10]:
b.state_size

LSTMStateTuple(c=32, h=32)

In [3]:
from migration.models.vrnn_keras import create_vrnn

In [ ]:
create_vrnn()

In [11]:
import sonnet as snt

a = snt.nets.MLP(
        output_sizes=[4, 8])

In [14]:
a.

ValueError: MLP(output_sizes=[4, 8]) does not currently contain any variables.

Most Sonnet modules create variables the first time they are called with an
input and requesting variables before this typically indicates a coding error.

You should refactor your code such that you request module variables after you
pass an example input to the module. For example:

    module = MLP(output_sizes=[4, 8])
    output = module(input)
    params = module.variables

If the module is stateless consider using `snt.allow_empty_variables(module)` to
suppress this error:

    module = MLP(output_sizes=[4, 8])
    snt.allow_empty_variables(module)
    params = module.variables

You can annotate your own subclasses directly if you prefer:

    @snt.allow_empty_variables
    class MyStatelessModule(snt.Module):
      pass

In [ ]:
from tensorflow.keras import layers
import tensorflow_probability as tfp

class ConditionalNormalDistribution(layers.Layer):
  """A Normal distribution conditioned on Tensor inputs via a fc network."""

  def __init__(self, size : int, hidden_layer_size : int, sigma_min : int =0.0,
               raw_sigma_bias=0.25):
    """Creates a conditional Normal distribution.

    Args:
      size: The dimension of the random variable.
      hidden_layers_size: The size of the hidden layers of the fully connected
        network used to condition the distribution on the inputs.
      sigma_min: The minimum standard deviation allowed, a scalar.
      raw_sigma_bias: A scalar that is added to the raw standard deviation
        output from the fully connected network. Set to 0.25 by default to
        prevent standard deviations close to 0.
    """
    super().__init__()
    
    # Trainable parameters
    self.dense1 = layers.Dense(hidden_layer_size, activation="relu")
    self.dense2 = layers.Dense(2*size, activation=None)

    # static parameters
    self.sigma_min = sigma_min
    self.raw_sigma_bias = raw_sigma_bias



  def _estimate_mu_and_sigma(self, inputs : tf.Tensor) -> tuple[tf.Tensor, tf.Tensor]:
    """Computes the parameters of a normal distribution based on the inputs."""

    # predict mu and sigma
    outs = self.dense1(inputs)
    outs = self.dense2(outs)
    mu, sigma = tf.split(outs, 2, axis=1)
    # correct
    sigma = tf.maximum(tf.nn.softplus(sigma + self.raw_sigma_bias), self.sigma_min)
    return mu, sigma
  
  def call(self, inputs : tf.Tensor):
    """Creates a normal distribution conditioned on the inputs."""
    mu, sigma = self._estimate_mu_and_sigma(inputs)
    return tfp.distributions.Normal(loc=mu, scale=sigma)


af = ConditionalNormalDistribution(4,4)
a = tf.convert_to_tensor([[1,2,3,4], [4,5,6,7]])
b = af(a)

In [83]:
b = ConditionalBernoulliDistribution(3,5,0.2)

In [81]:
a = tf.convert_to_tensor([[1,2,3,4], [4,5,6,7]])

ValueError: Exception encountered when calling ConditionalNormalDistribution.call().

[1mInvalid dtype: TrackedList[0m

Arguments received by ConditionalNormalDistribution.call():
  • inputs=tf.Tensor(shape=(1, 4), dtype=int32)
  • kwargs=<class 'inspect._empty'>